# NLP Mastery Journey — Module 9: Retrieval-Augmented Generation (RAG) & LLM Application Patterns

Module 8 ended on a decision framework: prompting, fine-tuning, or RAG. This module is RAG, built completely from parts you already have — the embeddings from Module 4, the semantic search you already wrote, and a prompt-construction step on top.

**A note on how this notebook is written**: you asked for the code to be extra beginner-friendly, so this module leans further into that than earlier ones — more small steps instead of dense one-liners, more `print()` statements showing you what's happening at each stage, plainer variable names, and more comments explaining *why*, not just what. Read the prints as you go; they're there so you can watch the pipeline work, step by step, instead of trusting it blindly.

### What this notebook teaches
| # | Topic | Why it matters |
|---|-------|------------------|
| 1 | What RAG actually is, and why | The problem it solves that neither prompting nor fine-tuning solves alone |
| 2 | Chunking documents | How you break long text into retrievable pieces, and why chunk size matters |
| 3 | Building a simple retriever, step by step | The full "search my documents" pipeline, in plain, readable code |
| 4 | Building the augmented prompt | Turning retrieved chunks into something an LLM can actually use |
| 5 | The generation step | Where a real LLM call plugs in |
| 6 | Evaluating RAG quality | Retrieval metrics AND generation metrics — they measure different things |
| 7 | Advanced RAG: re-ranking, hybrid search, query rewriting | The upgrades real production RAG systems add |
| 8 | Vector databases in production | Where FAISS/Pinecone/Weaviate fit in, and why |
| 9 | Beyond RAG: agents, tool use, structured output | The wider landscape of how LLM products are actually built |

### How to use this notebook
- Parts 1–4 (chunking, retrieval, prompt-building) run **completely live**, on a tiny example document, using only NumPy/pandas/scikit-learn — nothing to install, nothing to download.
- The final LLM call (Part 5) needs a real API key and internet, so it's shown correctly but commented out — exactly like earlier modules.
- **🔀 Alternatives** and **📋 Copy-paste template** callouts continue as before.


## 0. Setup

In [ ]:
# Nothing needs installing for Parts 1-4 - they only use libraries you
# already have from earlier modules (numpy, pandas, scikit-learn).
#
# For the PRODUCTION versions shown later (commented out), you would run:
# %pip install sentence-transformers faiss-cpu anthropic

import numpy as np
import pandas as pd

print("Setup done. No installs needed for the live parts of this notebook.")


## Part 1 — What RAG Actually Is, and Why

An LLM's knowledge is frozen at the moment it was trained, and it has never seen YOUR private documents (your company's internal docs, your own notes, today's news). Two things you already know how to do fix this in a specific combination:

1. **Retrieval** (Module 4): given a question, find the most relevant pieces of text from a document collection, using embeddings + similarity search.
2. **Generation**: hand those relevant pieces to an LLM, along with the original question, and let it write an answer USING that information.

That combination — retrieve, then generate — is the entire idea behind RAG. In plain words: **"look it up, then answer using what you found"** — instead of the model trying to answer purely from what it memorized during training.

### The RAG pipeline, in order
```
Your documents
      |
      v
Step 1: Split documents into small chunks         (Part 2)
      |
      v
Step 2: Turn each chunk into a vector (embedding)   (Part 3, reusing Module 4)
      |
      v
Step 3: Store all those vectors somewhere searchable (Part 3)
      |
      v
   ... time passes, a user asks a question ...
      |
      v
Step 4: Turn the QUESTION into a vector too          (Part 3)
Step 5: Find the chunks whose vectors are most similar (Part 3)
Step 6: Build a prompt: "question" + "here's what I found" (Part 4)
Step 7: Send that prompt to an LLM, get the answer back      (Part 5)
```
Every step below builds one piece of this pipeline, in order.


## Part 2 — Chunking: Splitting Documents Into Searchable Pieces

You can't search a whole 50-page document as one unit — it's too big and too unfocused to match well against a specific question. Instead, you split every document into small **chunks**, and search over the chunks individually.


In [ ]:
# ── Our example document (a small one, so you can see every chunk clearly) ──
# In a real project this would be loaded with the techniques from Module 1
# (a PDF, a scraped webpage, a .docx file, etc.) — here we just write it out
# directly so the whole notebook runs with zero setup.

example_document = """
Photosynthesis is the process plants use to turn sunlight into energy.
It happens mainly in the leaves, inside structures called chloroplasts.
Chlorophyll, the green pigment in chloroplasts, absorbs sunlight.
The plant combines that light energy with water and carbon dioxide.
This produces glucose, which the plant uses as food, and oxygen, which is released into the air.
Photosynthesis is the reason plants are sometimes called producers.
Animals cannot make their own food, so they depend on producers like plants.
This is why photosynthesis is considered the base of most food chains on Earth.
"""

print(example_document)


In [ ]:
# ── Chunking method 1: fixed-size chunking (simplest, most common starting point) ──
# We just cut the text into pieces of N words each. Simple, predictable, but
# it can awkwardly cut a sentence in half if you're not careful.

def chunk_by_word_count(text, chunk_size=25):
    """
    Split `text` into a list of chunks, each with (up to) `chunk_size` words.

    Step-by-step:
    1. Turn the whole text into one long list of words.
    2. Walk through that list, taking `chunk_size` words at a time.
    3. Join each group of words back into a chunk of text.
    """
    # Step 1: get every word in the document, in order
    all_words = text.split()

    # Step 2 and 3: build up the list of chunks
    chunks = []
    start = 0
    while start < len(all_words):
        end = start + chunk_size
        words_in_this_chunk = all_words[start:end]
        chunk_text = " ".join(words_in_this_chunk)
        chunks.append(chunk_text)
        start = end   # move the window forward for the next chunk

    return chunks

word_chunks = chunk_by_word_count(example_document, chunk_size=25)

print(f"Number of chunks created: {len(word_chunks)}")
print()
for i, chunk in enumerate(word_chunks):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()


In [ ]:
# ── Chunking method 2: sentence-based chunking (usually better for text quality) ──
# Instead of cutting mid-sentence, we group whole SENTENCES together. This
# tends to keep each chunk more coherent and easier for a model to use well.

def chunk_by_sentences(text, sentences_per_chunk=2):
    """
    Step-by-step:
    1. Split the text into individual sentences (using "." as a simple splitter).
    2. Group every `sentences_per_chunk` sentences into one chunk.
    """
    # Step 1: split on periods, then clean up whitespace, and drop empty pieces
    raw_sentences = text.split(".")
    sentences = []
    for sentence in raw_sentences:
        cleaned = sentence.strip()
        if cleaned:   # skip empty strings (e.g. from a trailing period)
            sentences.append(cleaned + ".")

    # Step 2: group sentences together, `sentences_per_chunk` at a time
    chunks = []
    start = 0
    while start < len(sentences):
        end = start + sentences_per_chunk
        group_of_sentences = sentences[start:end]
        chunk_text = " ".join(group_of_sentences)
        chunks.append(chunk_text)
        start = end

    return chunks

sentence_chunks = chunk_by_sentences(example_document, sentences_per_chunk=2)

print(f"Number of chunks created: {len(sentence_chunks)}")
print()
for i, chunk in enumerate(sentence_chunks):
    print(f"--- Chunk {i} ---")
    print(chunk)
    print()

# We'll use these SENTENCE-based chunks for the rest of the notebook, since
# they read more naturally.
document_chunks = sentence_chunks


In [ ]:
# 🔀 Chunk size: the most important knob in a RAG system, and a genuine trade-off
#
# | Chunk size | Pro                                     | Con                                        |
# |--------------|---------------------------------------------|-------------------------------------------------|
# | Small chunks   | Very focused, precise matches to a question    | May lose surrounding context a good answer needs   |
# | Large chunks     | Keeps more context together                       | Less focused matches; wastes prompt space on irrelevant text|
#
# A very common real-world compromise: use a MODERATE chunk size (a few
# hundred words), AND let neighboring chunks OVERLAP by a small amount
# (e.g. the last 1-2 sentences of one chunk are repeated at the start of the
# next) — this way, information near a chunk boundary doesn't get awkwardly
# split away from its context.
#
# 📋 COPY-PASTE TEMPLATE — overlapping chunks:
# def chunk_with_overlap(words, chunk_size=200, overlap=50):
#     chunks = []
#     start = 0
#     while start < len(words):
#         end = start + chunk_size
#         chunks.append(" ".join(words[start:end]))
#         start = end - overlap   # step BACK by `overlap` words before the next chunk,
#                                 # so the two chunks share some text
#     return chunks

print("Chunk-size guidance and an overlapping-chunk template shown above.")


## Part 3 — Building a Simple Retriever, Step by Step

Now we turn each chunk into a vector (an "embedding"), and later turn a QUESTION into a vector the same way, so we can measure which chunks are most similar to the question. For this live demo we use **TF-IDF** (Module 3) as our "embedding" — it needs zero downloads and is easy to inspect. A real production system would use a proper dense embedding model instead (shown as a template at the end of this part) — the PIPELINE shape is identical either way.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Step 1: turn every chunk into a vector ───────────────────────────────────
# fit_transform LEARNS the vocabulary from our chunks AND converts them to
# vectors, in one step (exactly like Module 3).

vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(document_chunks)

print(f"Number of chunks: {chunk_vectors.shape[0]}")
print(f"Vector size (vocabulary size): {chunk_vectors.shape[1]}")


In [ ]:
# ── Step 2: write a function that finds the best-matching chunks for a question ──
from sklearn.metrics.pairwise import cosine_similarity

def find_relevant_chunks(question, vectorizer, chunk_vectors, document_chunks, top_k=2):
    """
    Given a question, return the `top_k` chunks that are most SIMILAR to it.

    Step-by-step:
    1. Turn the question into a vector, using the SAME vectorizer as the chunks
       (this is important - the question and the chunks must be measured in
       the same "space" for the comparison to make sense).
    2. Compare the question's vector against EVERY chunk's vector.
    3. Sort the chunks by similarity score, highest first.
    4. Return just the top `top_k` chunks.
    """
    # Step 1: vectorize the question (use .transform, NOT .fit_transform -
    # we don't want to relearn the vocabulary, just reuse it, same rule as Module 3)
    question_vector = vectorizer.transform([question])

    # Step 2: compute a similarity score between the question and EVERY chunk
    similarity_scores = cosine_similarity(question_vector, chunk_vectors)[0]
    # similarity_scores is now a list of numbers, one per chunk -
    # higher number = more similar to the question

    # Step 3: figure out which chunks had the highest scores
    # np.argsort sorts from LOWEST to HIGHEST, so we reverse it with [::-1]
    # to get HIGHEST first
    ranked_chunk_indices = np.argsort(similarity_scores)[::-1]

    # Step 4: take only the top_k best chunks, and return them with their scores
    best_chunks = []
    for index in ranked_chunk_indices[:top_k]:
        chunk_text = document_chunks[index]
        chunk_score = similarity_scores[index]
        best_chunks.append((chunk_text, chunk_score))

    return best_chunks

# ── Let's try it! ─────────────────────────────────────────────────────────
question = "why do animals depend on plants?"
top_chunks = find_relevant_chunks(question, vectorizer, chunk_vectors, document_chunks, top_k=2)

print(f"Question: {question}")
print()
print("Most relevant chunks found:")
for chunk_text, score in top_chunks:
    print(f"  (similarity score: {score:.3f})  {chunk_text}")


In [ ]:
# 📋 COPY-PASTE TEMPLATE — the SAME retriever pattern, using REAL dense
# embeddings instead of TF-IDF (the production upgrade, from Module 4):
#
# from sentence_transformers import SentenceTransformer
# embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
#
# # Step 1: embed all chunks ONCE, and reuse this forever (don't re-embed on every question!)
# chunk_embeddings = embedding_model.encode(document_chunks)
#
# def find_relevant_chunks_with_embeddings(question, embedding_model, chunk_embeddings, document_chunks, top_k=2):
#     question_embedding = embedding_model.encode([question])
#     similarity_scores = cosine_similarity(question_embedding, chunk_embeddings)[0]
#     ranked_indices = np.argsort(similarity_scores)[::-1]
#     return [(document_chunks[i], similarity_scores[i]) for i in ranked_indices[:top_k]]
#
# 🔀 TF-IDF vs. dense embeddings for retrieval
# | Situation                                          | Better choice        |
# |--------------------------------------------------------|-------------------------|
# | Question uses the EXACT same words as the document        | TF-IDF works surprisingly well, and is fast/cheap|
# | Question uses DIFFERENT words with the SAME meaning          | Dense embeddings win - they capture meaning, not just exact words|
# | (e.g. document says "automobile", question says "car")          | Dense embeddings understand these are related; TF-IDF does not|

print("Dense-embedding retriever template shown above - same pipeline shape, better matching.")


## Part 4 — Building the Augmented Prompt

"Augmented" just means: we take the retrieved chunks and **add them into the prompt**, so the LLM has that information available when it writes its answer. This is the actual, simple mechanism behind the whole "RAG" idea — there's no magic beyond careful prompt construction here.


In [ ]:
def build_rag_prompt(question, retrieved_chunks):
    """
    Build the final text we'll send to the LLM. It has three clear parts:
    1. Instructions for the model (how to behave)
    2. The retrieved context (what we found)
    3. The actual question

    Step-by-step:
    - Join all the retrieved chunk texts together, clearly separated.
    - Wrap everything in a template that tells the model exactly what to do
      with the context we're giving it.
    """
    # Step 1: pull just the TEXT out of our (text, score) pairs, and number them
    context_pieces = []
    for i, (chunk_text, score) in enumerate(retrieved_chunks):
        context_pieces.append(f"[Source {i+1}]: {chunk_text}")
    context_block = "\n\n".join(context_pieces)

    # Step 2: build the final prompt using a clear template
    prompt = f"""Answer the question using ONLY the information in the sources below.
If the sources don't contain the answer, say so honestly instead of guessing.

Sources:
{context_block}

Question: {question}

Answer:"""

    return prompt

final_prompt = build_rag_prompt(question, top_chunks)
print(final_prompt)


In [ ]:
# 🔀 A few real prompt-template choices that matter in practice
#
# - "Answer using ONLY the sources" -> reduces the model making things up
#   (hallucinating) using outside knowledge instead of your actual documents.
# - "If the sources don't contain the answer, say so" -> makes the model
#   admit uncertainty instead of confidently guessing - very important for
#   any production system where wrong-but-confident answers are costly.
# - Numbering the sources ([Source 1], [Source 2]...) makes it easy to ALSO
#   ask the model to CITE which source it used for each part of its answer -
#   a common, valuable feature in real RAG products.

print("Prompt-design guidance shown above - small wording changes here make a real difference.")


## Part 5 — The Generation Step: Calling an LLM

The final step: send `final_prompt` to an LLM, and let it write the answer. This needs a real API key and internet, so it's shown as a correct, ready-to-use template rather than something we run live here.


In [ ]:
# import anthropic
#
# client = anthropic.Anthropic(api_key="your-api-key-here")   # better: load this
#                                                              # from an environment
#                                                              # variable, never hardcode it
#
# response = client.messages.create(
#     model="claude-sonnet-5",
#     max_tokens=500,
#     messages=[
#         {"role": "user", "content": final_prompt}   # the exact prompt we built in Part 4
#     ]
# )
#
# answer = response.content[0].text
# print(answer)

print("LLM call pattern shown above - this is the FULL RAG round trip:")
print("  question -> retrieve chunks -> build prompt -> call LLM -> get grounded answer")


## Part 6 — Evaluating RAG Quality

A RAG system can fail in TWO separate places, and you need to check both separately:
1. **Retrieval failure**: the right information exists in your documents, but the retriever didn't find it.
2. **Generation failure**: the retriever found the right chunks, but the LLM still gave a bad answer (ignored the context, hallucinated, misread it).

### Retrieval metrics (do we find the right chunks?)
- **Precision@k**: of the top-k chunks retrieved, what fraction are actually relevant?
- **Recall@k**: of ALL the relevant chunks that exist, what fraction did we find in the top-k?
- **MRR (Mean Reciprocal Rank)**: how high up the ranking was the FIRST relevant result? (Rewards getting a good chunk near the top, not just somewhere in the list.)

### Generation metrics (given the right chunks, did the LLM answer well?)
- **Faithfulness**: does the answer only say things actually SUPPORTED by the retrieved chunks? (Catches hallucination.)
- **Answer relevancy**: does the answer actually address the question that was asked?
- These are usually measured with an **LLM-as-judge** approach today (asking a strong model to score the answer against the source and the question) — the `RAGAS` library is a popular, purpose-built tool for exactly this.


In [ ]:
# ── A small, from-scratch example: computing Precision@k and Recall@k ───────

def precision_at_k(retrieved_chunk_indices, relevant_chunk_indices, k):
    """
    retrieved_chunk_indices: the indices of chunks we actually retrieved, in order
    relevant_chunk_indices: the indices we (or a human) know are TRULY relevant
    """
    top_k_retrieved = retrieved_chunk_indices[:k]
    num_relevant_in_top_k = 0
    for index in top_k_retrieved:
        if index in relevant_chunk_indices:
            num_relevant_in_top_k += 1
    return num_relevant_in_top_k / k

def recall_at_k(retrieved_chunk_indices, relevant_chunk_indices, k):
    top_k_retrieved = retrieved_chunk_indices[:k]
    num_relevant_found = 0
    for index in top_k_retrieved:
        if index in relevant_chunk_indices:
            num_relevant_found += 1
    return num_relevant_found / len(relevant_chunk_indices)

# Example: suppose chunks 2 and 5 are the TRUE relevant ones for some question,
# and our retriever returned chunks [2, 0, 5, 1] in that order:
example_retrieved = [2, 0, 5, 1]
example_truly_relevant = {2, 5}

print("Precision@2:", precision_at_k(example_retrieved, example_truly_relevant, k=2))
print("Recall@2:   ", recall_at_k(example_retrieved, example_truly_relevant, k=2))
print("Precision@4:", precision_at_k(example_retrieved, example_truly_relevant, k=4))
print("Recall@4:   ", recall_at_k(example_retrieved, example_truly_relevant, k=4))


## Part 7 — Advanced RAG: Re-ranking, Hybrid Search, Query Rewriting

Once the basic pipeline works, these three upgrades are the ones real production RAG systems add most often.

### Re-ranking
Retrieve MORE candidates than you need (e.g. top 20) using a fast method (like our TF-IDF search), then use a slower but MORE ACCURATE model (a "cross-encoder") to re-score just those 20 and pick the final top 3-5. This two-stage approach balances speed (the first pass is cheap) with quality (the second pass is more careful, but only run on a small candidate set).

### Hybrid search
Combine TWO retrieval methods and merge their results: **keyword search** (like TF-IDF, or a classic search engine like Elasticsearch — great at exact term matches, like product codes or names) PLUS **dense embedding search** (great at matching MEANING even with different wording). Neither method alone is best for everything; combining them is a very common real-world default.

### Query rewriting
Sometimes the user's raw question isn't the best possible search query (too short, ambiguous, or conversational). An extra LLM call can rewrite the question into a better search query BEFORE retrieval happens — e.g. expanding "it" or "that" in a follow-up question into what they actually refer to, using the conversation history.


In [ ]:
# 📋 COPY-PASTE TEMPLATE - simple hybrid search: combine TWO ranked lists
# by AVERAGING each chunk's rank across both methods (a simple, effective
# way to combine rankings without needing to tune complicated weights)

def hybrid_rank(keyword_search_ranking, embedding_search_ranking, all_chunk_indices):
    """
    Each input is a list of chunk indices, ordered BEST first.
    We convert each ranking into a SCORE (better rank = higher score),
    then add the two scores together for each chunk.
    """
    scores = {}
    for chunk_index in all_chunk_indices:
        scores[chunk_index] = 0

    # Give points based on position in the keyword ranking (higher = better rank)
    for position, chunk_index in enumerate(keyword_search_ranking):
        points = len(keyword_search_ranking) - position
        scores[chunk_index] += points

    # Do the same for the embedding-based ranking
    for position, chunk_index in enumerate(embedding_search_ranking):
        points = len(embedding_search_ranking) - position
        scores[chunk_index] += points

    # Sort chunks by their COMBINED score, best first
    combined_ranking = sorted(scores.keys(), key=lambda idx: scores[idx], reverse=True)
    return combined_ranking

# Example: two different search methods disagree somewhat on the best order
keyword_ranking = [2, 0, 1, 3]
embedding_ranking = [0, 2, 3, 1]
final_ranking = hybrid_rank(keyword_ranking, embedding_ranking, all_chunk_indices=[0, 1, 2, 3])
print("Combined hybrid ranking:", final_ranking)


## Part 8 — Vector Databases in Production

Everything above used a tiny in-memory list of chunks — fine for learning, not fine for a real document collection with millions of chunks that changes over time. Production RAG systems use a dedicated **vector database** instead, which adds things our simple demo doesn't have:
- **Fast approximate search** at huge scale (Module 4's FAISS, or a hosted option)
- **Persistence** (the index survives a server restart — ours lives only in memory)
- **Incremental updates** (add/remove documents without rebuilding the whole index from scratch)
- **Metadata filtering** (e.g. "only search documents from this customer" or "only documents from the last 30 days," combined with the similarity search)

| Option | Good for... |
|--------|----------------|
| **FAISS** (Module 4) | Self-hosted, you manage persistence/scaling yourself, maximum control |
| **Pinecone, Weaviate, Qdrant** | Managed vector databases — handle scaling/persistence/filtering for you |
| **pgvector** | You're already on PostgreSQL and want vector search without a new system |

The RETRIEVAL LOGIC you wrote in Part 3 doesn't fundamentally change when you swap in a real vector database — you're still doing "embed the query, find the most similar vectors" — the database just makes that operation fast and durable at real scale.


## Part 9 — Beyond RAG: Agents, Tool Use, and Structured Output

RAG is one pattern within a bigger picture of how real LLM products get built. Three more patterns worth knowing the names and shapes of:

### Tool use (function calling)
Instead of (or in addition to) retrieving documents, the model can be given a list of **tools** it's allowed to call (a calculator, a search API, a database query, code execution) and decides for itself when to use one, based on the question. RAG's retrieval step is really a special case of this: "search my documents" is just one possible tool among many.

### Agents
A model that can take MULTIPLE steps toward a goal — call a tool, look at the result, decide what to do next, call another tool, and so on — rather than answering in one single shot. This is how more complex tasks ("research this topic and write a report," "debug this failing test") get built on top of an LLM.

### Structured output
Instead of free-form text, you ask the model to return its answer in a specific, parseable format (JSON matching a schema you define). Extremely common in production, since downstream code usually needs to reliably parse the model's output, not just display it to a human.

These three patterns, plus RAG, are the actual building blocks behind most real LLM-based products today — including agentic coding tools and assistants.


## Part 10 — Full RAG Pipeline, Copy-Paste Ready

Putting every piece from this notebook together into one function you can drop into a project.


In [ ]:
def answer_question_with_rag(question, vectorizer, chunk_vectors, document_chunks, top_k=2):
    """
    📋 COPY-PASTE TEMPLATE - the whole RAG pipeline in one function.

    Step-by-step (matching every part of this notebook):
    1. Find the most relevant chunks for the question (Part 3).
    2. Build a prompt that includes those chunks (Part 4).
    3. Send the prompt to an LLM and return its answer (Part 5).
    """
    # Step 1: retrieve
    relevant_chunks = find_relevant_chunks(question, vectorizer, chunk_vectors, document_chunks, top_k=top_k)

    # Step 2: build the prompt
    prompt = build_rag_prompt(question, relevant_chunks)

    # Step 3: generate (commented out here since it needs a real API key)
    # response = client.messages.create(
    #     model="claude-sonnet-5",
    #     max_tokens=500,
    #     messages=[{"role": "user", "content": prompt}],
    # )
    # answer = response.content[0].text
    # return answer

    # For now, just return the prompt so you can see exactly what WOULD be sent:
    return prompt

result = answer_question_with_rag("what does chlorophyll do?", vectorizer, chunk_vectors, document_chunks)
print(result)


## Recap & What's Next

You built a complete RAG pipeline, live, from chunking through retrieval through prompt construction — and understand exactly where a real LLM call and a real vector database slot in for production use. You also now know the vocabulary and shape of the bigger picture RAG sits inside: re-ranking, hybrid search, query rewriting, tool use, agents, and structured output.

### Try this before the next lesson
1. Swap in a longer document of your own (a Wikipedia article works well) and try both chunking methods from Part 2 — compare the retrieval results.
2. Write 3-4 test questions with a known correct chunk for each, and compute Precision@k / Recall@k (Part 6) for your retriever.
3. Try the hybrid-ranking function from Part 7 with your own two rankings and see how the combined order changes.

### Next lesson in your NLP mastery path
**Module 10: Advanced NLP Tasks — Summarization, Translation & Question Answering, and How to Evaluate Generated Text** — building these tasks with modern models, and the metrics (ROUGE, BLEU, BERTScore, and LLM-as-judge) used to actually measure whether generated text is any good, closing the loop on evaluation across this entire course.
